# Retail Market Basket Intelligence: Phase 2 - Data Preprocessing & Feature Engineering

**Author:** Karthik Yelugam | Data Analyst  
**Environment:** Python, Pandas, NumPy, SQLAlchemy, MySQL  

### 📌 Executive Summary & Phase Objective
This notebook represents **Phase 2** of the Market Basket Intelligence pipeline. Building upon the anomalies identified in Phase 1 (EDA), the objective here is to execute a rigorous data cleaning and transformation pipeline.

**Key Operations Performed:**
1. **Data Cleansing:** Treatment of missing values, duplicates, and cancelled transactions.
2. **Anomaly & Outlier Removal:** Statistical filtering of invalid prices and extreme quantities using the IQR method.
3. **Standardization:** Text formatting and DateTime conversion.
4. **Feature Engineering:** Deriving business metrics (`Revenue`, `Transaction Size`, `Product Frequency`).
5. **Matrix Transformation:** Transforming the cleansed transactional data into a Sparse Boolean Matrix (Basket) required for the Apriori algorithm, filtering out ultra-rare products to reduce computational noise.

**Output:** Two production-ready datasets (`cleaned_dataset.csv` and `apriori_dataset.csv`) exported for Phase 3.

In [1]:
# =============================================================================
# 0. IMPORT REQUIRED LIBRARIES & CONFIGURE SETTINGS
# =============================================================================

# Core Data Manipulation
import pandas as pd
import numpy as np

# Database Connectivity
from sqlalchemy import create_engine
import pymysql

# Global Pandas Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✔ Libraries loaded and environment configured successfully.")

✔ Libraries loaded and environment configured successfully.


---
## 1. Database Connection & Raw Data Extraction
Extracting the raw transaction table directly from the relational database to initiate the cleaning pipeline.

In [2]:
# =============================================================================
# 1.1 SECURE EXTRACTION FROM PRODUCTION DATABASE
# =============================================================================

# Database Credentials
username = "dm_team16"
password = "2o_hihiFeTRE"
host = "18.136.157.135"
database = "project_purchase_pattern_analysis"

try:
    # Initialize Engine & Extract Data
    engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")
    print("Executing extraction query... Please wait.")
    df = pd.read_sql("SELECT * FROM mytable", engine)
    
    print(f"✔ Raw dataset loaded successfully! Initial Shape: {df.shape[0]:,} Rows, {df.shape[1]} Columns.")
    
    # Capture initial state for data quality tracking
    initial_rows = len(df)
    
except Exception as e:
    print(f"❌ Database connection failed: {e}")

Executing extraction query... Please wait.
✔ Raw dataset loaded successfully! Initial Shape: 522,064 Rows, 7 Columns.


---
## 2. Missing Value Treatment
Addressing critical gaps in the data. Products missing `Itemname` cannot be analyzed in Apriori and must be dropped. Missing `CustomerID` values are preserved using a "Guest_Customer" placeholder since Market Basket Analysis relies primarily on `BillNo`.

In [3]:
# =============================================================================
# 2.1 HANDLE MISSING PRODUCT NAMES & CUSTOMER IDs
# =============================================================================

rows_before_missing = len(df)

# 1. Target Missing Product Names (Itemname)
# Convert empty strings to NaN, then drop rows missing critical product identifiers
df["Itemname"] = df["Itemname"].replace("", np.nan)
df = df.dropna(subset=["Itemname"])

# 2. Target Missing Customer Identifiers
# Convert empty strings to NaN, then fill with a professional placeholder
df["CustomerID"] = df["CustomerID"].replace("", np.nan)
df["CustomerID"] = df["CustomerID"].fillna("Guest_Customer")

print(f"✔ Missing Value Treatment Complete.")
print(f"  Rows Removed (Missing Itemnames): {rows_before_missing - len(df):,}")
print(f"  Current Dataset Shape: {df.shape}")

✔ Missing Value Treatment Complete.
  Rows Removed (Missing Itemnames): 1,455
  Current Dataset Shape: (520609, 7)


---
## 3. Data Deduplication & Anomaly Removal
Removing exact duplicate rows, cancelled transactions (denoted by "C" in BillNo), and logically invalid data points (negative quantities implying returns, and negative/zero pricing).

In [4]:
# =============================================================================
# 3.1 REMOVE DUPLICATES, CANCELLATIONS, AND LOGICAL ERRORS
# =============================================================================

rows_before_anomalies = len(df)

# 1. Remove Exact Duplicate Rows
duplicates_count = df.duplicated().sum()
df = df.drop_duplicates()

# 2. Filter out Cancelled Transactions
# Cancellations do not represent successful basket co-occurrence
df = df[~df["BillNo"].astype(str).str.startswith("C")]

# 3. Remove Invalid Quantities (Returns/Errors)
df = df[df["Quantity"] > 0]

# 4. Remove Invalid Pricing (Zero or negative)
df = df[df["Price"] > 0]

print(f"✔ Anomaly Filtering Complete.")
print(f"  Duplicates Removed: {duplicates_count:,}")
print(f"  Total Anomalous Rows Filtered: {rows_before_anomalies - len(df):,}")
print(f"  Current Dataset Shape: {df.shape}")

✔ Anomaly Filtering Complete.
  Duplicates Removed: 5,286
  Total Anomalous Rows Filtered: 6,339
  Current Dataset Shape: (514270, 7)


---
## 4. Statistical Outlier Handling (IQR Method)
Extreme outliers in `Quantity` and `Price` can severely skew revenue metrics and represent bulk wholesale purchases rather than standard retail consumer baskets. We isolate the core consumer base using the Interquartile Range (IQR).

In [5]:
# =============================================================================
# 4.1 FILTER OUTLIERS USING INTERQUARTILE RANGE (IQR)
# =============================================================================

rows_before_outliers = len(df)

def remove_outliers(dataframe, column):
    """Utility function to filter extreme outliers based on IQR boundaries."""
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    return dataframe[(dataframe[column] >= lower_bound) & (dataframe[column] <= upper_bound)]

# Apply IQR filtering to Quantity and Price
df = remove_outliers(df, "Quantity")
df = remove_outliers(df, "Price")

print(f"✔ Outlier Treatment Complete.")
print(f"  Extreme Outlier Rows Removed: {rows_before_outliers - len(df):,}")
print(f"  Current Dataset Shape: {df.shape}")

✔ Outlier Treatment Complete.
  Extreme Outlier Rows Removed: 84,143
  Current Dataset Shape: (430127, 7)


---
## 5. Text Standardization & Temporal Formatting
Ensuring strict uniformity across string columns to prevent the Apriori algorithm from treating "Product A " and "PRODUCT A" as different items.

In [6]:
# =============================================================================
# 5.1 STANDARDIZE TEXT AND PARSE DATETIME
# =============================================================================

# 1. Clean Product Names (Strip whitespaces and format to UPPERCASE)
df["Itemname"] = df["Itemname"].astype(str).str.strip().str.upper()

# 2. Clean Country Names (Strip whitespaces and format to Title Case)
df["Country"] = df["Country"].astype(str).str.strip().str.title()

# 3. Convert Present_Date string to highly optimized DateTime format
df["Present_Date"] = pd.to_datetime(df["Present_Date"], format="%d-%m-%Y %H:%M", errors="coerce")

print("✔ Text and DateTime Formatting Complete.")
display(df[["Itemname", "Country", "Present_Date"]].head())

✔ Text and DateTime Formatting Complete.


,Itemname,Country,Present_Date
0,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom,2010-12-01 08:26:00
1,WHITE METAL LANTERN,United Kingdom,2010-12-01 08:26:00
2,CREAM CUPID HEARTS COAT HANGER,United Kingdom,2010-12-01 08:26:00
3,KNITTED UNION FLAG HOT WATER BOTTLE,United Kingdom,2010-12-01 08:26:00
4,RED WOOLLY HOTTIE WHITE HEART.,United Kingdom,2010-12-01 08:26:00


---
## 6. Feature Engineering
Enriching the dataset with calculated metrics required for downstream BI dashboards and deep analytics. We engineer `Revenue`, `Transaction Size`, `Product Frequency`, and distinct temporal features.

In [7]:
# =============================================================================
# 6.1 DERIVE BUSINESS & TEMPORAL METRICS
# =============================================================================

# Calculate Financial Revenue per line item
df["Revenue"] = df["Quantity"] * df["Price"]

# Calculate Transaction Size (Items per Basket)
transaction_size = df.groupby("BillNo")["Itemname"].count().reset_index(name="Transaction_Size")
df = df.merge(transaction_size, on="BillNo", how="left")

# Calculate Universal Product Frequency
product_frequency = df["Itemname"].value_counts().reset_index()
product_frequency.columns = ["Itemname", "Product_Frequency"]
df = df.merge(product_frequency, on="Itemname", how="left")

# Extract Temporal Features for Trend Analysis
df['Year'] = df['Present_Date'].dt.year
df["Month"] = df["Present_Date"].dt.month
df["Day"] = df["Present_Date"].dt.day
df["Hour"] = df["Present_Date"].dt.hour
df["Day_of_Week"] = df["Present_Date"].dt.day_name()

print("✔ Feature Engineering Complete. New features successfully mapped.")
display(df[["BillNo", "Itemname", "Revenue", "Transaction_Size", "Product_Frequency", "Month", "Hour"]].head())

✔ Feature Engineering Complete. New features successfully mapped.


,BillNo,Itemname,Revenue,Transaction_Size,Product_Frequency,Month,Hour
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,15.300,7,1886,12,8
1,536365,WHITE METAL LANTERN,20.340,7,287,12,8
2,536365,CREAM CUPID HEARTS COAT HANGER,22.000,7,265,12,8
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,20.340,7,422,12,8
4,536365,RED WOOLLY HOTTIE WHITE HEART.,20.340,7,354,12,8


---
## 7. Apriori Basket Matrix Engineering
The Apriori algorithm requires a specialized matrix where each row represents a unique transaction (basket), and each column represents a unique product. Values must be strictly binary (1 if purchased, 0 if not).

In [8]:
# =============================================================================
# 7.1 CREATE BOOLEAN SPARSE MATRIX
# =============================================================================

print("Pivoting data into Basket Matrix... Please wait.")

# Pivot the table: BillNo as rows, Itemname as columns, filled with counts
basket = (
    df.groupby(["BillNo", "Itemname"])["Itemname"]
    .count()
    .unstack()
    .fillna(0)
)

# Convert integer counts to purely binary values (0 or 1). 
# Note: Using highly optimized Pandas boolean casting instead of deprecated applymap()
basket_matrix = (basket > 0).astype(int)

print(f"✔ Basket Matrix successfully generated!")
print(f"  Total Unique Baskets (Transactions): {basket_matrix.shape[0]:,}")
print(f"  Total Unique Products (Features): {basket_matrix.shape[1]:,}")

Pivoting data into Basket Matrix... Please wait.
✔ Basket Matrix successfully generated!
  Total Unique Baskets (Transactions): 17,597
  Total Unique Products (Features): 3,765


---
## 8. Rare Product Filtering
Including products that are rarely purchased creates computational noise and generates insignificant, non-actionable association rules. To ensure highly relevant business insights, we filter out products that appear fewer than 100 times across all transactions.

In [9]:
# =============================================================================
# 8.1 FILTER ULTRA-RARE PRODUCTS FROM THE MATRIX
# =============================================================================

# Identify products with support >= 100 purchases
frequent_products = basket_matrix.sum(axis=0)
frequent_products = frequent_products[frequent_products >= 100].index

# Filter the matrix to retain only frequent products
final_apriori_dataset = basket_matrix[frequent_products]

print(f"✔ Rare Product Filtering Complete.")
print(f"  Products reduced from {basket_matrix.shape[1]:,} down to {final_apriori_dataset.shape[1]:,} high-frequency items.")
print(f"  Final Apriori Dataset Shape: {final_apriori_dataset.shape}")

✔ Rare Product Filtering Complete.
  Products reduced from 3,765 down to 1,295 high-frequency items.
  Final Apriori Dataset Shape: (17597, 1295)


---
## 9. Final Validation & Data Export
Validating the structural integrity of both the cleaned Master Dataset and the Apriori Matrix before exporting them for Phase 3.

In [10]:
# =============================================================================
# 9.1 DATASET EXPORT TO LOCAL STORAGE
# =============================================================================

import os

# Create local destination directories if they don't exist
os.makedirs('data/processed', exist_ok=True)

# Export the master cleaned dataset (For BI and Statistical Analysis)
clean_file_path = "data/processed/cleaned_dataset.csv"
df.to_csv(clean_file_path, index=False)

# Export the boolean matrix dataset (For the Apriori Algorithm)
apriori_file_path = "data/processed/apriori_dataset.csv"
final_apriori_dataset.to_csv(apriori_file_path, index=True) # Must keep index=True to preserve BillNo

print("=================================================================")
print("🎯 PHASE 2 PIPELINE SUCCESSFULLY COMPLETED!")
print("=================================================================")
print(f"Initial Raw Rows        : {initial_rows:,}")
print(f"Final Cleaned Rows      : {len(df):,}")
print(f"Cleaned File Saved To   : {clean_file_path}")
print(f"Apriori File Saved To   : {apriori_file_path}")
print("\nProceed to '03_apriori_market_basket_analysis.ipynb' to initiate Association Rule Mining.")

🎯 PHASE 2 PIPELINE SUCCESSFULLY COMPLETED!
Initial Raw Rows        : 522,064
Final Cleaned Rows      : 430,127
Cleaned File Saved To   : data/processed/cleaned_dataset.csv
Apriori File Saved To   : data/processed/apriori_dataset.csv

Proceed to '03_apriori_market_basket_analysis.ipynb' to initiate Association Rule Mining.
